In [ ]:
import numpy as np
import importlib
from scipy.integrate import solve_ivp
import ra_utilities as util

importlib.reload(util)

# Ring Attractor Atchitetchure

Much like a Hopfeild network, a ring attractor is a fully connected recurrent neural network. What differentiates it from Hopfeild networks are the additional symetry constraints on its connectivity matrix. Ring attractor networks require a symetric (specifically circulant) autoassociative matrix $A$ to maintian the bump at any location on the manifold and an asymetric heteroassociative matrix $H$ to move the bump ridgidly around the manifold. We will discuss the requirements for these matricies in more depth on the board but we will use the general form of equations from Noorman et al., 2024 in the code below:

### $ A_{ij} = J_I + J_E (cos(\Theta_j - \Theta_k)) $

Where $J_E$ and $J_I$ respectively control the strength of recurrent excitation and inhibition in the connectvitiy between neurons in the network and $A_{ij}$ describes the weight between neurons with prefered orientations $\Theta_j$ and $\Theta_k$. We then have:

### $ H_{ij} = sin(\Theta_j - \Theta_k) $

Which describes the asymetric shifting matrix required for angular path integration in the network and importantly satisfies $ H = A' $, the form nesissary for ridgid movement of the activity profile. 

The overall connectivity in the network is then given by:

### $ W = A + v(t) H $

Where $v(t)$ describes the current angular velocity (in radians) with negitive values taken to mean counterclockwise. 


# Network dynamics

The netowrk evolves according to the neural feild equation, a differential equation broadly used to simulate the dynamics of recurrent neural networks:

### $ \tau \frac{\delta u}{\delta t} = -u + W f(u) + I(t) $

With $f(x)$ taken to be a threshold linear unit here: $f(x < thr) = 0$, $f(x ≥ thr) = x$ and $I(t)$ taken to be some time-varying feed forward input to the network (left as zero in the simulations below). Here $u$ and $I(t)$ are taken to be vectors of length N and $W$ is the full weight matrix of dimensionality NxN. $u_i$ describes the actviation of neruon $i$ and $f(u_i)$ gives the firing rate of neruon $i$.

First, let's initialize a network of N neurons and solve for the optimal values of $J_I$ and $J_E$:

In [ ]:
N = 16
Nact = 8
JE = util.optimal_JE(N, Nact)
print('JE* =', JE)
JI = util.find_JI_for_amplitude(N, JE, target_amplitude=0.2, n_samples=300)
print('JI =', JI)

# Let's visualize the connection matricies for our network

In [ ]:
print(f"Symmetric {N}x{N} weight matrix:")
util.plot_weight_matrix(N, JE, JI, symmetric=True)

print(f"\nAsymmetric {N}x{N} weight matrix:")
util.plot_weight_matrix(N, JE, JI, symmetric=False)

# Let's test that our bump is persistent: that it stays where we leave it when there is no velocity input

Try chaning the value of `psi0` and checking that this holds for arbitrary positions.

In [ ]:
# no-input drift test: bump should roughly persist at psi0
t, h = util.simulate_bump(N, JE, JI, vin=0.0, t_max=5.0, psi0=0.3, animate=True)
psi = util.bump_orientation(h)
print('no-input orientation start/end:', psi[0], psi[-1])

# Now, let's check that the bump moves smoothly with a constant velocity input without getting stuck anywhere or changing shape

Try changing the input velocity `vin` (sign and magnitude) as well as the starting location of the bump `psi0` and seeing weather this changes the dynamics. 

In [ ]:
# velocity integration test: bump should move under constant vin
t, h = util.simulate_bump(N, JE, JI, vin=0.5, t_max=3.0, psi0=0.0, animate=True)
psi = util.bump_orientation(h)
print('velocity-driven orientation (last 5):', psi[-5:])
print('amplitude (last 5):', util.bump_amplitude(h)[-5:])

# Adding noise

### Let's now add small, random perterbations to our weight matricies and see how it affects the persistency and path integration capabilities of the network. 

`rho` discribes the range of noise values, try playing arounf with it and seeing how it affects the dynamics. Also try leaving `rho` as is and changing the size of the network N, what happens?

In [ ]:
rng = np.random.default_rng(0)
JE_noise = util.sample_weight_noise(N, rho=0.5, rng=rng)
JI_noise = util.sample_weight_noise(N, rho=0.5, rng=rng)

util.plot_weight_matrix(N, JE, JI, symmetric=True, JE_noise=JE_noise, JI_noise=JI_noise)

t, h = util.simulate_bump(N, JE, JI, vin=0.0, t_max=5.0, psi0=0.2, JE_noise=JE_noise, JI_noise=JI_noise, animate=True)

### Now, lets examine how the path integration is affected for the same network:

In [ ]:
t, h = util.simulate_bump(N, JE, JI, vin=0.5, t_max=5.0, psi0=0.2, JE_noise=JE_noise, JI_noise=JI_noise, animate=True)